<a href="https://colab.research.google.com/github/Esteban-Paz2024/Plantilla-Proyecto-ABP/blob/main/docs/notebooks/getting_started_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Getting Started - RocketPy in Colab


We start by setting up our environment. To run this notebook, we will need:

- RocketPy
- Data files (we will download RocketPy's data from github)

Therefore, let's run the following lines of code:

In [ ]:
!pip install rocketpy
!curl -o NACA0012-radians.txt https://raw.githubusercontent.com/RocketPy-Team/RocketPy/master/data/airfoils/NACA0012-radians.txt
!curl -o Cesaroni_M1670.eng https://raw.githubusercontent.com/RocketPy-Team/RocketPy/master/data/motors/cesaroni/Cesaroni_M1670.eng
!curl -o powerOffDragCurve.csv https://raw.githubusercontent.com/RocketPy-Team/RocketPy/master/data/rockets/calisto/powerOffDragCurve.csv
!curl -o powerOnDragCurve.csv https://raw.githubusercontent.com/RocketPy-Team/RocketPy/master/data/rockets/calisto/powerOnDragCurve.csv

Now we can start!

Here we go through a simplified rocket trajectory simulation to get you started. Let's start by importing the rocketpy module.


In [ ]:
from rocketpy import Environment, Flight, Rocket, SolidMotor

It is recommended to run the following line to make matplotlib plots which will be shown later interactive and higher quality.


In [ ]:
%config InlineBackend.figure_formats = ['svg']
%matplotlib inline

## Setting Up a Simulation


### Creating an Environment for Spaceport America



The `Environment` class is used to define the atmosphere, the winds, and the gravity models.

You can find more information about the `Environment` class in the [Environment Class Usage Docs](https://docs.rocketpy.org/en/latest/notebooks/environment/environment_class_usage.html).

In [ ]:
env = Environment(latitude=32.990254, longitude=-106.974998, elevation=1400)

To get weather data from the GFS forecast, available online, we run the following lines.

First, we set tomorrow's date.


In [ ]:
import datetime

tomorrow = datetime.date.today() + datetime.timedelta(days=1)

env.set_date(
    (tomorrow.year, tomorrow.month, tomorrow.day, 12)
)  # Hour given in UTC time

Then, we tell env to use a GFS forecast to get the atmospheric conditions for flight.

Don't mind the warning, it just means that not all variables, such as wind speed or atmospheric temperature, are available at all altitudes given by the forecast.


In [ ]:
env.set_atmospheric_model(type="Forecast", file="GFS")

We can see what the weather will look like by calling the info method!


In [ ]:
env.info()

### Creating a Motor

A solid rocket motor is used in this case. To create a motor, the SolidMotor class is used and the required arguments are given.

The SolidMotor class requires the user to have a thrust curve ready. This can come either from a .eng file for a commercial motor, such as below, or a .csv file from a static test measurement.

Besides the thrust curve, other parameters such as grain properties and nozzle dimensions must also be given.

See [Solid Motor Class Usage Docs](https://docs.rocketpy.org/en/latest/user/motors/solidmotor.html) for more information.


In [ ]:
Pro75M1670 = SolidMotor(
    thrust_source="Cesaroni_M1670.eng",
    dry_mass=1.815,
    dry_inertia=(0.125, 0.125, 0.002),
    nozzle_radius=33 / 1000,
    grain_number=5,
    grain_density=1815,
    grain_outer_radius=33 / 1000,
    grain_initial_inner_radius=15 / 1000,
    grain_initial_height=120 / 1000,
    grain_separation=5 / 1000,
    grains_center_of_mass_position=0.397,
    center_of_dry_mass_position=0.317,
    nozzle_position=0,
    burn_time=3.9,
    throat_radius=11 / 1000,
    coordinate_system_orientation="nozzle_to_combustion_chamber",
)

**Pay special attention to *position* related parameters:**
More details on [Positions and Coordinate Systems](https://docs.rocketpy.org/en/latest/user/positions.html)


To see what our thrust curve looks like, along with other import properties, we invoke the info method yet again. You may try the all_info method if you want more information all at once!


In [ ]:
Pro75M1670.info()

### Creating a Rocket


A rocket is composed of several components. Namely, we must have a motor (good thing we have the Pro75M1670 ready), a couple of aerodynamic surfaces (nose cone, fins and tail) and parachutes (if we are not launching a missile).

You can find more information about the `Rocket` class in the [Rocket Class Usage Docs](https://docs.rocketpy.org/en/latest/user/rocket.html).

Let's start by initializing our rocket, named Calisto, entering inertia properties, some dimensions and drag curves.

**Pay special attention to *position* related parameters:**
More details on [Positions and Coordinate Systems](https://docs.rocketpy.org/en/latest/user/positions.html)



In [ ]:
calisto = Rocket(
    radius=127 / 2000,
    mass=14.426,
    inertia=(6.321, 6.321, 0.034),
    power_off_drag="powerOffDragCurve.csv",
    power_on_drag="powerOnDragCurve.csv",
    center_of_mass_without_motor=0,
    coordinate_system_orientation="tail_to_nose",
)

rail_buttons = calisto.set_rail_buttons(
    upper_button_position=0.0818,
    lower_button_position=-0.618,
    angular_position=45,
)

To add the motor to our rocket we need only inform what motor we are adding (Pro75M1670) and inform the position, in meters, of the motor's nozzle exit area relative to the previously defined coordinate system.


In [ ]:
calisto.add_motor(Pro75M1670, position=-1.255)

#### Adding Aerodynamic Surfaces


Now we define the aerodynamic surfaces. They are really straight forward with special attention needed only for the position values. Here is a quick guide:

- The positions given **must** be relative to the same coordinate system as the rockets;
- Position of the Nosecone refers to the tip of the nose;
- Position of fins refers to the point belonging to the root chord which is highest in the rocket coordinate system;
- Position of the tail the point belonging to the tail which is highest in the rocket coordinate system.

See more details in [Positions and Coordinate Systems](https://docs.rocketpy.org/en/latest/user/positions.html)

In [ ]:
nose_cone = calisto.add_nose(length=0.55829, kind="vonKarman", position=1.278)

fin_set = calisto.add_trapezoidal_fins(
    n=4,
    root_chord=0.120,
    tip_chord=0.060,
    span=0.110,
    position=-1.04956,
    cant_angle=0.5,
    airfoil=("NACA0012-radians.txt", "radians"),
)

tail = calisto.add_tail(
    top_radius=0.0635, bottom_radius=0.0435, length=0.060, position=-1.194656
)

To see all information regarding the rocket we just defined we run:


In [ ]:
calisto.all_info()

#### Adding Parachutes


Finally, we have parachutes! Calisto will have two parachutes, Drogue and Main. The Drogue parachute will open at apogee while the Main parachute will open at 800m above ground level

For more details see [Adding Parachutes](https://docs.rocketpy.org/en/latest/user/rocket.html#adding-parachutes)

In [ ]:
Main = calisto.add_parachute(
    "Main",
    cd_s=10.0,
    trigger=800,
    sampling_rate=105,
    lag=1.5,
    noise=(0, 8.3, 0.5),
)

Drogue = calisto.add_parachute(
    "Drogue",
    cd_s=1.0,
    trigger="apogee",
    sampling_rate=105,
    lag=1.5,
    noise=(0, 8.3, 0.5),
)

Just be careful if you run this last cell multiple times! If you do so, your rocket will end up with lots of parachutes which activate together, which may cause problems during the flight simulation. We advise you to re-run all cells which define our rocket before running this, preventing unwanted old parachutes. Alternatively, you can run the following lines to remove parachutes.

```python
Calisto.parachutes.remove(Drogue)
Calisto.parachutes.remove(Main)
```


## Simulating a Flight

Simulating a flight trajectory is as simple as initializing a Flight class object givin the rocket and environnement set up above as inputs. The launch rail inclination and heading are also given here.


In [ ]:
test_flight = Flight(
    rocket=calisto, environment=env, rail_length=5.2, inclination=85, heading=0
)

## Analyzing the Results

RocketPy gives you many plots, thats for sure! They are divided into sections to keep them organized. Alternatively, see the Flight class documentation to see how to get plots for specific variables only, instead of all of them at once.


In [ ]:
test_flight.all_info()

Export Flight Trajectory to a .kml file so it can be opened on Google Earth


In [ ]:
from rocketpy.simulation import FlightDataExporter

FlightDataExporter(test_flight).export_kml(
    file_name="trajectory.kml",
    extrude=True,
    altitude_mode="relative_to_ground",
)

## 3D Flight Animation

RocketPy can render an interactive 3D animation of the rocket's trajectory and attitude using [PyVista](https://pyvista.org/). This feature requires the optional `animation` extra:

```bash
pip install rocketpy[animation]
```

> **Note:** A local desktop Python or Jupyter environment provides the best interactive experience. Google Colab requires a compatible PyVista Jupyter backend.

Two animation modes are available:

| Method | What it shows |
|---|---|
| `flight.plots.animate_trajectory()` | Simulated and flown paths, scalar coloring, telemetry, velocity and wind, flight events and georeferenced ground imagery |
| `flight.plots.animate_rotate()` | Centred attitude view with body, velocity and wind vectors, stability markers, an inertial reference sphere and live angular rates |

Both views provide play/pause, a draggable flight-time control, 0.5x/1x/2x/3x playback speeds, optional kinematic charts, camera presets and deterministic GIF/MP4 export. Day/night colors are selected from local launch time and blend toward near-space navy with altitude. The trajectory defaults to speed coloring and also supports Mach, dynamic pressure, acceleration and altitude. The rotate view can show aerodynamic angles, 3-1-3 Euler angles, body rates, center of mass and center of pressure. Pass `color_scheme` to override the shared palette. The methods accept an optional `file_name` argument pointing to a custom `.stl` model. When omitted, RocketPy uses a built-in default rocket shape.

In [ ]:
# Install the optional animation dependency (skip if already installed)
!pip install "rocketpy[animation]"

# Animate the full trajectory â€” rocket moves through 3D space
# Press Escape or close the window to exit the animation
test_flight.plots.animate_trajectory(
    start=0,
    stop=test_flight.t_final,
    time_step=0.05,
    color_by="speed",
    show_kinematic_plots=True,
    camera_mode="follow",
)

# Alternatively, animate attitude and stability diagnostics
# test_flight.plots.animate_rotate(
#     start=0,
#     stop=test_flight.t_final,
#     time_step=0.05,
#     show_attitude_plots=True,
#     show_cp_cm=True,
# )

# Deterministic export:
# test_flight.plots.animate_trajectory(export_file="flight.mp4", export_fps=30)

# To use your own 3D model, pass its path via file_name:
# test_flight.plots.animate_trajectory(file_name="my_rocket.stl")

## Flight results (Python implementation):

Graphs of the simulated flight in OpenRocket, reconstructed from the exported CSV files.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
RUTA = "/content/drive/MyDrive/AESS/"

### Movement:

The rocket reaches an apogee of around 1,350 m in approximately 16 seconds, with a maximum speed of ~185 m/s when thrust ceases. After deployment, the descent is almost linear, at a virtually constant speed. This is the base graph: it plots all the events used in the other graphs against time.

In [ ]:
import matplotlib.pyplot as plt

tiempo =[]
altitud =[]
velocidad = []
aceleracion = []
eventos = {}

with open(RUTA + "vuelo1_sim2.csv", "r", encoding="utf-8") as f:
    for linea in f:
        linea = linea.strip()
        if not linea:
            continue

        if linea.startswith("# Event"):
            partes = linea.replace("# Event ", "").split(" occurred at t=")
            nombre = partes[0].strip()
            t_evento = float(partes[1].replace(" seconds", "").strip())
            eventos[nombre] = {"t": t_evento, "idx": len(tiempo)}
            continue

        if linea.startswith("#"):
            continue
        try:
            partes=linea.split(",")
            tiempo.append(float(partes[0]))
            altitud.append(float(partes[1]))
            velocidad.append(float(partes[2]))
            aceleracion.append(float(partes[3]))
        except (ValueError, IndexError):
            continue

corte_t = eventos["RECOVERY_DEVICE_DEPLOYMENT"]["t"]
corte = eventos["RECOVERY_DEVICE_DEPLOYMENT"]["idx"]

colores_evento = {
    "LIFTOFF": ("green", "Despegue"),
    "LAUNCHROD": ("pink", "Encendido del motor"),
    "BURNOUT": ("red", "Apagado del motor"),
    "EJECTION_CHARGE": ("violet", "Carga de ejeccion"),
    "APOGEE": ("black", "Apogeo"),
    "GROUND_HIT": ("blue", "Aterrizaje del vuelo")
}

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 14), sharey=True)

for ax, ini, fin in [(ax1, 0, corte), (ax2, corte, None)]:
    ax.plot(tiempo[ini:fin], altitud[ini:fin], color="steelblue", linewidth=2.0, label="Altitud de vuelo (m)")
    ax.plot(tiempo[ini:fin], velocidad[ini:fin], color="green", linewidth=1.5, label="Velocidad total (m/s)")
    ax.plot(tiempo[ini:fin], aceleracion[ini:fin], color="orange", linewidth=1.5, label="Aceleración total (m/s²)")
    ax.grid(True, linestyle="--", alpha=0.5)

ax1.set_title("Previo al despliegue")
ax2.set_title("Después del despliegue")

ax1.set_xlim(tiempo[0], tiempo[corte - 1])
ax2.set_xlim(tiempo[corte], tiempo[-1])

ax1.set_ylabel("Movimiento (m, m/s, m/s²)")
ax2.set_ylabel("Movimiento (m, m/s, m/s²)")
ax2.set_xlabel("Tiempo (s)")

for nombre, info in eventos.items():
    if nombre in colores_evento:
        color, etiqueta = colores_evento[nombre]
        ax = ax1 if info["t"] <= corte_t else ax2
        ax.axvline(x=info["t"], color=color, linestyle="--", linewidth=1.2, alpha=0.5, label=etiqueta)

handles, labels = ax1.get_legend_handles_labels()
for h, l in zip(*ax2.get_legend_handles_labels()):
    if l not in labels:
        handles.append(h); labels.append(l)
unicos = dict(zip(labels, handles))

fig.suptitle("Movimiento registrado durante el vuelo del cohete", fontsize=14, y=0.98, x=0.45)
fig.legend(unicos.values(), unicos.keys(), loc="upper right", ncol=2)

plt.tight_layout(rect=[0, 1, 1, 0.95])
plt.show()

### Mass and propulsion:

The total mass decreases from ~9.2 to ~8.3 kg as the propellant is consumed. The initial thrust-to-weight ratio (T/W) is around 17–18, well above the minimum recommended for a safe launch from the launch pad.

In [ ]:
import matplotlib.pyplot as plt

velocidad = []
masa_total = []
masa_propulsor = []
empuje_peso = []
eventos = {}

with open(RUTA + "vuelo1_sim6.csv", "r", encoding="utf-8") as f:
    for linea in f:
        linea = linea.strip()

        if not linea:
            continue

        if linea.startswith("# Event"):
            try:
                partes = linea.replace("# Event ", "").split(" occurred at t=")
                nombre = partes[0].strip()
                t_evento = float(partes[1].replace(" seconds", "").strip())
                eventos[nombre] = {
                    "t": t_evento,
                    "idx": len(velocidad)
                }
            except (ValueError, IndexError):
                pass
            continue

        if linea.startswith("#"):
            continue

        try:
            partes = linea.split(",")
            v = float(partes[0])
            m_total = float(partes[1])
            m_prop = float(partes[2])
            tw = float(partes[3])

            velocidad.append(v)
            masa_total.append(m_total)
            masa_propulsor.append(m_prop)
            empuje_peso.append(tw)

        except (ValueError, IndexError):
            continue

if not velocidad:
    raise ValueError("No se encontraron datos válidos en vuelo1_sim6.csv.")

inicio = eventos.get("LIFTOFF", {"idx": 0})["idx"]
fin = eventos.get("BURNOUT", {"idx": len(velocidad)})["idx"]
inicio = max(0, min(inicio, len(velocidad) - 1))
fin = max(inicio + 1, min(fin, len(velocidad)))

velocidad_graf = velocidad[inicio:fin]
masa_total_graf = masa_total[inicio:fin]
masa_propulsor_graf = masa_propulsor[inicio:fin]
empuje_peso_graf = empuje_peso[inicio:fin]

colores_evento = {
    "LIFTOFF": ("green", "Despegue"),
    "LAUNCHROD": ("orange", "Abandono de guía"),
    "BURNOUT": ("red", "Apagado del motor"),
    "APOGEE": ("purple", "Apogeo"),
    "GROUND_HIT": ("black", "Aterrizaje"),
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6.5))

# Se conservan los datos en orden temporal.
# NO uses sorted(velocidad) ni sorted(empuje_peso).
ax1.plot(
    velocidad,
    masa_total,
    color="steelblue",
    linewidth=2.0,
    label="Masa total del cohete"
)

ax1.plot(
    velocidad,
    masa_propulsor,
    color="tomato",
    linewidth=2.0,
    label="Masa de propelente"
)

ax2.plot(
    empuje_peso,
    masa_total,
    color="steelblue",
    linewidth=2.0,
    label="Masa total del cohete"
)

ax2.plot(
    empuje_peso,
    masa_propulsor,
    color="tomato",
    linewidth=2.0,
    label="Masa de propelente"
)

# Marcadores de los eventos usando el valor real de cada variable.
for nombre, info in eventos.items():
    if nombre not in colores_evento:
        continue

    idx = min(info["idx"], len(velocidad) - 1)

    if idx < 0:
        continue

    color, etiqueta = colores_evento[nombre]

    ax1.scatter(
        velocidad[idx],
        masa_total[idx],
        color=color,
        s=60,
        zorder=5,
        label=etiqueta
    )

    ax2.scatter(
        empuje_peso[idx],
        masa_total[idx],
        color=color,
        s=60,
        zorder=5,
        label=etiqueta
    )

# Formato: masa frente a velocidad
ax1.set_title("Masa del cohete frente a velocidad", fontsize=13)
ax1.set_xlabel("Velocidad total (m/s)", fontsize=12)
ax1.set_ylabel("Masa (kg)", fontsize=12)
ax1.grid(True, linestyle="--", alpha=0.5)

# Formato: masa frente a relación empuje-peso
ax2.set_title("Masa del cohete frente a relación empuje-peso", fontsize=13)
ax2.set_xlabel("Relación empuje-peso, T/W (-)", fontsize=12)
ax2.set_ylabel("Masa (kg)", fontsize=12)
ax2.grid(True, linestyle="--", alpha=0.5)

# Leyenda sin entradas repetidas
handles, labels = ax1.get_legend_handles_labels()
leyenda = dict(zip(labels, handles))

fig.legend(
    leyenda.values(),
    leyenda.keys(),
    loc="upper center",
    ncol=4,
    bbox_to_anchor=(0.5, 1.02)
)

fig.suptitle(
    "Variación de la masa durante el vuelo completo",
    fontsize=15,
    y=1.08
)

plt.tight_layout()
plt.show()

### Aerodynamic drag:

The drag coefficient remains between ~0.51 and ~0.58 across the Mach number range (up to ~0.54). Friction decreases with increasing Mach number, whilst pressure drag increases. This determines the extent to which the rocket is slowed down and, consequently, its apogee.

In [ ]:
import matplotlib.pyplot as plt

mach        = []
cd_total    = []
cd_friccion = []
cd_presion  = []
eventos     = {}

with open(RUTA + "vuelo1_sim1.csv", "r", encoding="utf-8") as f:
    for linea in f:
        linea = linea.strip()
        if not linea:
            continue

        if linea.startswith("# Event"):
            partes = linea.replace("# Event ", "").split(" occurred at t=")
            nombre = partes[0].strip()
            t_evento = float(partes[1].replace(" seconds", "").strip())
            eventos[nombre] = {"t": t_evento, "idx": len(mach)}
            continue

        if linea.startswith("#"):
            continue

        try:
            partes = linea.split(",")
            cd  = float(partes[0])
            cdf = float(partes[1])
            cdp = float(partes[2])
            mac = float(partes[3])

            if cd < 5 and mac > 0.005:
                cd_total.append(cd)
                cd_friccion.append(cdf)
                cd_presion.append(cdp)
                mach.append(mac)
        except (ValueError, IndexError):
            continue

print(f"Puntos válidos tras filtro: {len(mach)}")
print(f"Mach mín: {min(mach):.4f}  |  Mach máx: {max(mach):.4f}")
print(f"Coef.rozamiento mín: {min(cd_total):.4f}  |  Coef. rozamiento máx: {max(cd_total):.4f}")

# ─── GRAFICAR ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 6))

ax.plot(mach, cd_total, color="steelblue", linewidth=2.0, label="Coef. de rozamiento")
ax.plot(mach, cd_friccion, color="tomato", linewidth=1.5, label="Coef. por fricción")
ax.plot(mach, cd_presion, color="goldenrod", linewidth=1.5, label="Presión del coef. de rozamiento")

# Eventos de vuelo como líneas verticales
colores_evento = {
    "LIFTOFF":   ("green",  "Despegue"),
    "LAUNCHROD": ("orange", "Abandono guía"),
    "BURNOUT":   ("red",    "Apagado motor"),
    "APOGEE":    ("purple", "Apogeo"),
}

for nombre, info in eventos.items():
    if nombre in colores_evento:
        idx = info["idx"]
        # Buscar el Mach correspondiente al índice del evento en los datos originales
        # Usamos el Mach del primer punto válido más cercano al evento
        color, etiqueta = colores_evento[nombre]
        t = info["t"]
        ax.axvline(x=t * (max(mach) / max(e["t"] for e in eventos.values())),
                   color=color, linestyle="--", linewidth=1.2,
                   alpha=0.7, label=etiqueta)

ax.set_xlabel("Número de Mach", fontsize=12)
ax.set_ylabel("Coeficiente de rozamiento", fontsize=12)
ax.set_title("Simulation 11 — numero de Match vs. Coeficiente de rozamiento (vuelo ascendente)", fontsize=13)
ax.legend(loc="upper right")
ax.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

### Stability:

The centre of pressure (CP, ~170 cm) lies behind the centre of gravity (CG, ~143 cm). The stability margin remains at ~2–2.7 calibres throughout the flight; in other words, the rocket is stable.

In [ ]:
import matplotlib.pyplot as plt

tiempo = []
margen = []
cp = []
cg = []
eventos = {}

with open(RUTA + "vuelo1_sim3.csv", "r", encoding="utf-8") as f:
    for linea in f:
        linea = linea.strip()
        if not linea:
            continue

        if linea.startswith("# Event"):
            partes = linea.replace("# Event ", "").split(" occurred at t=")
            nombre = partes[0].strip()
            t_evento = float(partes[1].replace(" seconds", "").strip())
            eventos[nombre] = {"t": t_evento, "idx": len(tiempo)}
            continue

        if linea.startswith("#"):
            continue
        try:
            partes = linea.split(",")
            tiempo.append(float(partes[0]))
            cp.append(float(partes[1]))
            cg.append(float(partes[2]))
            margen.append(float(partes[3]))
        except (ValueError, IndexError):
            continue

corte = eventos["RECOVERY_DEVICE_DEPLOYMENT"]["idx"]
tiempo = tiempo[:corte]
margen = margen[:corte]
cp = cp[:corte]
cg = cg[:corte]

fig, ax = plt.subplots(figsize=(13,8))
ax.plot(tiempo, cp, color="steelblue", linewidth=2.0, label="Situacion del CP (cm)")
ax.plot(tiempo, cg, color="green", linewidth=1.5, label="Situacion del CG (cm)")
ax.plot(tiempo, margen, color="orange", linewidth=1.5, label="Margen de estabilidad en calibres")

colores_evento = {
    "LIFTOFF": ("green", "Despegue"),
    "LAUNCHROD": ("pink", "Encendido del motor"),
    "BURNOUT": ("red", "Apagado del motor"),
    "EJECTION_CHARGE": ("violet", "Carga de ejeccion"),
    "APOGEE": ("black", "Apogeo")
}

for nombre, info in eventos.items():
    if nombre in colores_evento:
        idx = info["idx"]
        color, etiqueta = colores_evento[nombre]
        t = info["t"]
        ax.axvline(x=t * (max(tiempo)/ max(e["t"] for e in eventos.values())), color=color,
                   linestyle="--", linewidth=1.2, alpha=0.5, label=etiqueta)

ax.set_xlabel("Tiempo (s)", fontsize=12)
ax.set_ylabel("Situación de Estabilidad del cohete (-, cm)", fontsize=12)
ax.set_title("Estabilidad del cohete", fontsize=13)
ax.legend(loc="upper right")
ax.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

### Direction:



#### Angles:

The angle of attack drops from 85° to almost 0° in approximately 0.5 seconds and remains low until near apogee. The zenith drops from approximately 85° to approximately 2° as the trajectory curves, whilst the azimuth remains virtually unchanged.

In [ ]:
import matplotlib.pyplot as plt

tiempo = []
a_ataque= []
o_vertical = []
o_horizontal = []
eventos = {}

with open(RUTA + "vuelo1_sim4.csv", "r", encoding="utf-8") as f:
    for linea in f:
        linea = linea.strip()
        if not linea:
            continue

        if linea.startswith("# Event"):
            partes = linea.replace("# Event ", "").split(" occurred at t=")
            nombre = partes[0].strip()
            t_evento = float(partes[1].replace(" seconds", "").strip())
            eventos[nombre] = {"t": t_evento, "idx": len(tiempo)}
            continue

        if linea.startswith("#"):
            continue
        try:
            partes = linea.split(",")
            tiempo.append(float(partes[0]))
            a_ataque.append(float(partes[1]))
            o_vertical.append(float(partes[2]))
            o_horizontal.append(float(partes[3]))
        except (ValueError, IndexError):
            continue

corte = eventos["RECOVERY_DEVICE_DEPLOYMENT"]["idx"]
tiempo = tiempo[:corte]
a_ataque = a_ataque[:corte]
o_vertical = o_vertical[:corte]
o_horizontal = o_horizontal[:corte]

fig, ax = plt.subplots(figsize=(13,8))
ax.plot(tiempo, a_ataque, color="steelblue", linewidth=2.0, label="Ángulo de ataque (°)")
ax.plot(tiempo, o_vertical, color="green", linewidth=1.5, label="Orientación vertical - zenit (°)")
ax.plot(tiempo, o_horizontal, color="orange", linewidth=1.5, label="Orientación horizontal - azimut (°)")

colores_evento = {
    "LIFTOFF": ("green", "Despegue"),
    "LAUNCHROD": ("pink", "Encendido del motor"),
    "BURNOUT": ("red", "Apagado del motor"),
    "EJECTION_CHARGE": ("violet", "Carga de ejeccion"),
    "APOGEE": ("black", "Apogeo")
}

for nombre, info in eventos.items():
    if nombre in colores_evento and info["t"] <= tiempo[-1]:
        color, etiqueta = colores_evento[nombre]
        ax.axvline(x=info["t"], color=color, linestyle="--", linewidth=1.2, alpha=0.5, label=etiqueta)

ax.set_xlabel("Tiempo (s)", fontsize=12)
ax.set_ylabel("Orientación vertical y horizontal (°)", fontsize=12)
ax.set_title("Orientación del cohete", fontsize=13)
ax.legend(loc="upper right")
ax.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

#### Rates:

The pitch rate fluctuates and is damped at the start, and increases near apogee because the rocket begins to tip over. The yaw is almost zero and the rotation is negligible (~10⁻⁷ °/s).

In [ ]:
import matplotlib.pyplot as plt

tiempo = []
t_rotacion = []
t_cabeceo = []
t_guinada = []
eventos = {}

with open(RUTA + "vuelo1_sim5.csv", "r", encoding="utf-8") as f:
    for linea in f:
        linea = linea.strip()
        if not linea:
            continue

        if linea.startswith("# Event"):
            partes = linea.replace("# Event ", "").split(" occurred at t=")
            nombre = partes[0].strip()
            t_evento = float(partes[1].replace(" seconds", "").strip())
            eventos[nombre] = {"t": t_evento, "idx": len(tiempo)}
            continue

        if linea.startswith("#"):
            continue
        try:
            partes = linea.split(",")
            tiempo.append(float(partes[0]))
            t_rotacion.append(float(partes[1]))
            t_cabeceo.append(float(partes[2]))
            t_guinada.append(float(partes[3]))
        except (ValueError, IndexError):
            continue

corte = eventos["RECOVERY_DEVICE_DEPLOYMENT"]["idx"]
tiempo = tiempo[:corte]
t_rotacion = t_rotacion[:corte]
t_cabeceo = t_cabeceo[:corte]
t_guinada = t_guinada[:corte]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 9), sharex=True, gridspec_kw={"height_ratios": [3, 1]})

ax1.plot(tiempo, t_cabeceo, color="green", linewidth=1.5, label="Tasa de cabeceo (°/s)")
ax1.plot(tiempo, t_guinada, color="orange", linewidth=1.5, label="Tasa de guiñada (°/s)")
ax1.set_ylabel("Tasa (°/s)", fontsize=12)
ax1.set_title("Tasas de la orientación del cohete", fontsize=13)
ax1.legend(loc="upper right")
ax1.grid(True, linestyle="--", alpha=0.5)

ax2.plot(tiempo, t_rotacion, color="steelblue", linewidth=1.2, label="Tasa de rotación (°/s)")
ax2.set_xlabel("Tiempo (s)", fontsize=12)
ax2.set_ylabel("Rotación (°/s)", fontsize=10)
ax2.legend(loc="upper right")
ax2.grid(True, linestyle="--", alpha=0.5)
ax2.ticklabel_format(axis="y", style="scientific", scilimits=(0, 0))

colores_evento = {
    "LIFTOFF": ("green", "Despegue"),
    "LAUNCHROD": ("pink", "Encendido del motor"),
    "BURNOUT": ("red", "Apagado del motor"),
    "EJECTION_CHARGE": ("violet", "Carga de ejeccion"),
    "APOGEE": ("black", "Apogeo")
}

for nombre, info in eventos.items():
    if nombre in colores_evento and info["t"] <= tiempo[-1]:
        color, etiqueta = colores_evento[nombre]
        ax1.axvline(x=info["t"], color=color, linestyle="--", linewidth=1.2, alpha=0.5, label=etiqueta)
        ax1.axvline(x=info["t"], color=color, linestyle="--", linewidth=1.2, alpha=0.5, label=etiqueta)

plt.tight_layout()
plt.show()

## Using Simulation for Design

Here, we go through a couple of examples which make use of RocketPy in cool ways to help us design our rocket.


### Apogee as a Function of Mass

This one is a classic one! We always need to know how much our rocket's apogee will change when our payload gets heavier.


In [ ]:
from rocketpy.utilities import apogee_by_mass

apogee_by_mass(flight=test_flight, min_mass=5, max_mass=20, points=10, plot=True)

### Out of Rail Speed as a Function of Mass

Lets make a really important plot. Out of rail speed is the speed our rocket has when it is leaving the launch rail. This is crucial to make sure it can fly safely after leaving the rail. A common rule of thumb is that our rocket's out of rail speed should be 4 times the wind speed so that it does not stall and become unstable.


In [ ]:
from rocketpy.utilities import liftoff_speed_by_mass

liftoff_speed_by_mass(flight=test_flight, min_mass=5, max_mass=20, points=10, plot=True)

### Dynamic Stability Analysis

Ever wondered how static stability translates into dynamic stability? Different static margins result in different dynamic behavior, which also depends on the rocket's rotational inertial.

Let's make use of RocketPy's helper class called Function to explore how the dynamic stability of Calisto varies if we change the fins span by a certain factor.


In [ ]:
# Helper class
import copy

from rocketpy import Function

# Prepare a copy of the rocket
calisto2 = copy.deepcopy(calisto)

# Prepare Environment Class
custom_env = Environment()
custom_env.set_atmospheric_model(type="custom_atmosphere", wind_v=-5)

# Simulate Different Static Margins by Varying Fin Position
simulation_results = []

for factor in [-0.5, -0.2, 0.1, 0.4, 0.7]:
    # Modify rocket fin set by removing previous one and adding new one
    calisto2.aerodynamic_surfaces.pop(-1)

    fin_set = calisto2.add_trapezoidal_fins(
        n=4,
        root_chord=0.120,
        tip_chord=0.040,
        span=0.100,
        position=-1.04956 * factor,
    )
    # Simulate
    print(
        "Simulating Rocket with Static Margin of {:1.3f}->{:1.3f} c".format(
            calisto2.static_margin(0),
            calisto2.static_margin(calisto2.motor.burn_out_time),
        )
    )
    test_flight = Flight(
        rocket=calisto2,
        environment=custom_env,
        rail_length=5.2,
        inclination=90,
        heading=0,
        max_time_step=0.01,
        max_time=5,
        terminate_on_apogee=True,
        verbose=True,
    )
    # Store Results
    static_margin_at_ignition = calisto2.static_margin(0)
    static_margin_at_out_of_rail = calisto2.static_margin(test_flight.out_of_rail_time)
    static_margin_at_steady_state = calisto2.static_margin(test_flight.t_final)
    simulation_results += [
        (
            test_flight.attitude_angle,
            "{:1.2f} c | {:1.2f} c | {:1.2f} c".format(
                static_margin_at_ignition,
                static_margin_at_out_of_rail,
                static_margin_at_steady_state,
            ),
        )
    ]

Function.compare_plots(
    simulation_results,
    lower=0,
    upper=1.5,
    xlabel="Time (s)",
    ylabel="Attitude Angle (deg)",
)

### Characteristic Frequency Calculation

Here we analyse the characteristic frequency of oscillation of our rocket just as it leaves the launch rail. Note that when we ran test_flight.all_info(), one of the plots already showed us the frequency spectrum of our flight. Here, however, we have more control of what we are plotting.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Simulate first 5 seconds of Flight
flight = Flight(
    rocket=calisto,
    environment=env,
    rail_length=5.2,
    inclination=90,
    heading=0,
    max_time_step=0.01,
    max_time=5,
)

# Perform a Fourier Analysis
Fs = 100.0
# sampling rate
Ts = 1.0 / Fs
# sampling interval
t = np.arange(1, 400, Ts)  # time vector
ff = 5
# frequency of the signal
y = flight.attitude_angle(t) - np.mean(flight.attitude_angle(t))
n = len(y)  # length of the signal
k = np.arange(n)
T = n / Fs
frq = k / T  # two sides frequency range
frq = frq[range(n // 2)]  # one side frequency range
Y = np.fft.fft(y) / n  # fft computing and normalization
Y = Y[range(n // 2)]

# Create the plot
fig, ax = plt.subplots(2, 1)
ax[0].plot(t, y)
ax[0].set_xlabel("Time")
ax[0].set_ylabel("Signal")
ax[0].set_xlim((0, 5))
ax[0].grid()
ax[1].plot(frq, abs(Y), "r")  # plotting the spectrum
ax[1].set_xlabel("Freq (Hz)")
ax[1].set_ylabel("|Y(freq)|")
ax[1].set_xlim((0, 5))
ax[1].grid()
plt.subplots_adjust(hspace=0.5)
plt.show()